# improved_v3: trích xuất khái niệm y khoa từ bệnh án tiếng Việt

Kế thừa improved_v2/T1, thêm Phase 4 A1: assertion-lite isNegated rất bảo thủ bằng ConText proposer + hai Qwen verifier next-token logits; baseline v2 được giữ nguyên.

Notebook chạy trọn trên một **Colab T4 (16 GB)**: hai teacher nạp ở **4-bit**, compute dtype đặt `float16` vì T4 không hỗ trợ bfloat16 hiệu quả. Kiến trúc chi tiết ở `docs/02_method.md`.

## 1. Kiểm tra runtime

Runtime, Change runtime type, chọn **T4 GPU**.

In [ ]:
!nvidia-smi

## 2. Clone và cài đặt

Colab đã có torch bản CUDA. `pyproject.toml` là nguồn phụ thuộc duy nhất. Cài thêm `.[quant]` để nạp hai teacher ở 4-bit.

In [ ]:
!git clone https://github.com/AIVIETNAM-AIO-DinhBao/ViClinicalIE_2 medextract
%cd medextract
!pip install -e ".[quant]"    # bitsandbytes cho chế độ 4-bit

## 3. Self-check

Không cần GPU, không cần knowledge base. Phải in PASS cho cả bốn mục CONFIG / IMPORTS / SCHEMA / PATHS.

In [ ]:
!python scripts/selfcheck.py

## 4. Knowledge base cho bước linking

`improved_v2` chỉ dùng exact-alias lookup trên hai bảng parquet, **không** cần SapBERT và **không** cần FAISS index, nên chỉ cần hai lệnh build dưới đây. Danh mục ICD-10 tiếng Việt (TT06) **đã đi kèm repo** tại `data/kb/raw/`, nên bạn chỉ cần tải RxNorm và đặt vào `data/kb/raw/RXNCONSO.RRF` (xem `INSTALL.md` cho các nguồn RxNorm).

In [ ]:
from pathlib import Path

raw_dir = Path("data/kb/raw")
icd_file = raw_dir / "Phu_luc_Bang_danh_muc_ICD10_FINAL_TT06_2026.xlsx"
rxnorm_file = raw_dir / "RXNCONSO.RRF"
print("KB raw files:")
!ls -lh data/kb/raw
if not icd_file.exists():
    raise FileNotFoundError(f"Missing ICD-10 TT06 file from repo: {icd_file}")
if not rxnorm_file.exists():
    raise FileNotFoundError(
        "Missing RXNCONSO.RRF. This file is license-gated, so it is not committed to the public repo. "
        "Upload it to data/kb/raw/RXNCONSO.RRF or copy it from Drive before running build_rxnorm."
    )
print("OK: required KB raw files are available.")


In [ ]:
!python -m medextract.kb.build_icd    --tt06    # -> data/kb/processed/icd_terms_v2.parquet
!python -m medextract.kb.build_rxnorm --v2      # -> data/kb/processed/rxnorm_terms_v2.parquet

## 5. Chon checkpoint NER Phase 1

Mount Google Drive va tu tim checkpoint GLiNER fine-tuned Phase 1. Neu tim thay, bien `PHASE1_NER_MODEL` se duoc dung de override `ner.model` trong config chay submission.

Uu tien ban full: `models/gliner_phase1_full_alltxt_dev30/final`; fallback sang ban trial no-dev12 neu co.

In [ ]:
from pathlib import Path

try:
    from google.colab import drive
    drive.mount("/content/drive")
except ModuleNotFoundError:
    print("Not running inside Google Colab; assuming Drive/local paths are already available.")

PHASE1_NER_CANDIDATES = [
    # Best Phase 1 full run: clean train + full old all.txt silver + pseudo-dev30 eval.
    Path("/content/drive/Shareddrives/R2AI/Viettel_AI_Race/models/gliner_phase1_full_alltxt_dev30/final"),
    Path("/content/drive/MyDrive/Viettel_AI_Race/models/gliner_phase1_full_alltxt_dev30/final"),

    # Fallback: trial run excluding old doc_id 1..12 to avoid dev leakage.
    Path("/content/drive/Shareddrives/R2AI/Viettel_AI_Race/models/gliner_phase1_trial_no_dev12/final"),
    Path("/content/drive/MyDrive/Viettel_AI_Race/models/gliner_phase1_trial_no_dev12/final"),
]

def is_gliner_model_dir(path: Path) -> bool:
    return (
        path.is_dir()
        and (path / "gliner_config.json").exists()
        and ((path / "pytorch_model.bin").exists() or (path / "model.safetensors").exists())
    )

print("Phase 1 NER checkpoint candidates:")
for candidate in PHASE1_NER_CANDIDATES:
    status = "OK" if is_gliner_model_dir(candidate) else "missing/incomplete"
    print(f"- {candidate}: {status}")

PHASE1_NER_MODEL = next((p for p in PHASE1_NER_CANDIDATES if is_gliner_model_dir(p)), None)
if PHASE1_NER_MODEL is None:
    raise FileNotFoundError(
        "Could not find GLiNER Phase 1 checkpoint. "
        "Run the Phase 1 training notebook or copy the final folder to "
        "Drive: Viettel_AI_Race/models/gliner_phase1_full_alltxt_dev30/final"
    )

PHASE1_NER_MODEL = str(PHASE1_NER_MODEL)
print("Using Phase 1 NER model:", PHASE1_NER_MODEL)


## 6. Config phu cho Colab T4 + Phase 1 NER

Hai teacher nap 4-bit, compute dtype `float16`. Config nay ke thua `configs/improved_v2.yaml`, dong thoi override `ner.model` sang checkpoint GLiNER Phase 1 va dung `label_map` tieng Viet dung voi model fine-tuned.

In [ ]:
import pathlib

# Repo id cua hai teacher tren Hugging Face Hub.
# Neu ban da tai san trong so ve may, thay bang duong dan cuc bo.
PRIMARY_MODEL = "Qwen/Qwen3-4B-Instruct-2507"
SECONDARY_MODEL = "Qwen/Qwen3.5-4B"

if "PHASE1_NER_MODEL" not in globals():
    raise RuntimeError("Run the Phase 1 checkpoint cell before creating colab_t4.yaml")

override = f"""# Config phu cho Colab T4 + Phase 1 fine-tuned GLiNER NER.
# Ke thua improved_v2, override NER model/labels va quantization.
extends: configs/improved_v2.yaml

ner:
  model: "{PHASE1_NER_MODEL}"
  max_chunk_chars: 800
  raw_floor: 0.02
  label_map:
    "TRIỆU_CHỨNG": "TRIỆU_CHỨNG"
    "CHẨN_ĐOÁN": "CHẨN_ĐOÁN"
    "THUỐC": "THUỐC"
    "TÊN_XÉT_NGHIỆM": "TÊN_XÉT_NGHIỆM"
    "KẾT_QUẢ_XÉT_NGHIỆM": "KẾT_QUẢ_XÉT_NGHIỆM"
  thresholds:
    TRIỆU_CHỨNG: 0.20
    CHẨN_ĐOÁN: 0.25
    THUỐC: 0.30
    TÊN_XÉT_NGHIỆM: 0.15
    KẾT_QUẢ_XÉT_NGHIỆM: 0.35

consensus_selector:
  primary_model: {PRIMARY_MODEL}
  secondary_model: {SECONDARY_MODEL}
  primary_device: cuda:0
  secondary_device: cuda:0
  batch_size: 16

quantization:
  mode: 4bit
  compute_dtype: float16
  double_quant: true
"""

pathlib.Path("colab_t4.yaml").write_text(override, encoding="utf-8")
print(override)


## 7. Chay day du va dong goi ban nop

Notebook dung truc tiep 100 file `.txt` da co trong `data/input/`. `--zip` ghi `out/improved_v2/submission.zip`, cac file JSON nam phang, khong co thu muc con.

In [ ]:
from pathlib import Path

input_dir = Path("data/input")
txt_files = sorted(input_dir.glob("*.txt"), key=lambda p: int(p.stem) if p.stem.isdigit() else p.stem)
print(f"Found {len(txt_files)} input .txt files in {input_dir}")
print([p.name for p in txt_files[:10]])
if len(txt_files) != 100:
    raise RuntimeError(f"Expected 100 input files in {input_dir}, found {len(txt_files)}")

!rm -rf out/improved_v2
!python run.py --config colab_t4.yaml --input data/input \
               --output out/improved_v2 --zip

!echo "JSON outputs:"
!find out/improved_v2 -maxdepth 1 -type f -name "*.json" | wc -l
!ls -lh out/improved_v2/submission.zip


## 7b. Phase 4 A1: T1 + Qwen-confirmed negation assertions

Cell này giữ nguyên `colab_t4.yaml` baseline, tạo thêm `colab_t4_assertion_neg.yaml` chỉ bật `isNegated` rất bảo thủ: ConText propose cue phủ định, cả hai Qwen phải xác nhận bằng next-token logits margin >= 2.0. Sau khi chạy, dùng `scripts/audit_assertions.py` để kiểm tra tỷ lệ assertion không rỗng trước khi nộp.


In [ ]:
import pathlib

phase4 = """# Phase 4 A1: colab_t4 baseline + conservative Qwen-confirmed negation.
extends: colab_t4.yaml

assertions:
  neg_window_chars: 60
  block_lookback_lines: 6

assertion_selector:
  enabled: true
  labels: [isNegated]
  primary_margin: 2.0
  secondary_margin: 2.0
  context_window_chars: 180
"""
pathlib.Path("colab_t4_assertion_neg.yaml").write_text(phase4, encoding="utf-8")
print(phase4)

!rm -rf out/improved_v2_assertion_neg
!python run.py --config colab_t4_assertion_neg.yaml --input data/input \
               --output out/improved_v2_assertion_neg --zip

!python scripts/audit_assertions.py --pred out/improved_v2_assertion_neg --input data/input --samples 12
!ls -lh out/improved_v2_assertion_neg/submission.zip


## 8. Xem mot mau output

In [ ]:
import json, pathlib

p = pathlib.Path("out/improved_v2/001.json")
data = json.load(open(p, encoding="utf-8"))
print(f"{p.name}: {len(data)} concept(s)\n")
print(json.dumps(data[:3], ensure_ascii=False, indent=2))

## 8. Chấm điểm local

`score.py` là bản đọc lại công thức của Ban Tổ chức để xếp hạng hai lần chạy local, không phải bộ chấm chính thức. Chuẩn bị thư mục nhãn dạng `<thư mục nhãn>/{stem}.json` cùng schema với bản nộp, rồi:

```bash
python score.py --pred out/improved_v2 --gold <thư mục nhãn> -v
```